# 03. 입찰메이트 RFP RAG — Phase 2 로컬 파싱·출처 추적

유영관 · 2026-11-11

원본 PDF·HWP·HWPX를 **같은 로컬 처리 흐름**으로 읽고, 텍스트·표·문서 버전·원문 위치를 확인합니다.
이 노트북과 동봉 코드에는 OpenAI 호출, 외부 OCR 서비스 호출, GitHub 작업이 없습니다.

| 확인할 것 | 방법 |
| --- | --- |
| 실제 파일 형식 | 확장자 대신 원본 바이트와 컨테이너 검사 |
| 원본 버전 | 원본에서 다시 계산한 SHA256과 입력 목록 비교 |
| 본문과 표 | 문단·PDF 물리 페이지·표의 행/열·병합 범위와 출처 보관 |
| 평가 근거 | Phase 1의 인용을 새 파싱 결과에서 다시 찾기 |
| 처리 한계 | 파싱 실패·누락 의심·시각 검수·OCR 대상을 별도 기록 |

**처음에는 기본 설정으로 실행하세요.** 이미 원본을 새 파서로 처리한 결과가 동봉되어 있습니다.
전체 원본을 강제로 다시 파싱하는 옵션은 기본값 `False`이며, 켜면 PDF 표 처리 때문에 수 분이 걸릴 수 있습니다.
숫자·문자열 복구 검증은 RAG의 검색·답변 정확도 점수가 아닙니다.


## 1. 로컬 작업 폴더와 설정

ZIP을 푼 뒤 `notebooks/` 안에서 엽니다. 폴더를 옮겼다면 `PROJECT_ROOT`에 압축을 푼 경로를 직접 지정할 수 있습니다.
기본 확인은 표준 Python 3.10 이상에서 수행합니다. 원본 재파싱과 선택적 PDF 렌더/OCR에 필요한 라이브러리는 해당 작업을 켤 때만 사용합니다.
설치 명령이나 다운로드를 자동 실행하지 않습니다.


In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib.metadata
import json
import sys

PROJECT_ROOT = None  # 예: Path(r'C:\workspace\rfp_phase2_local_20261111')
REPARSE_FROM_ORIGINAL = False
RUN_PDF_RENDER = False
RUN_LOCAL_OCR = False

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent,
                  Path.cwd() / 'rfp_phase2_local_20261111',
                  Path.cwd() / 'output/rfp_phase2_local_20261111']
    ROOT = next((p.resolve() for p in candidates
                 if (p / 'data/input_manifest.json').is_file()
                 and (p / 'local_ingest/loader.py').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None:
    raise FileNotFoundError('압축을 푼 패키지 안에서 열거나 PROJECT_ROOT를 직접 지정하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from local_ingest.loader import (detect_format, load_documents, run_ingestion,
                                 safe_source_path, sha256_file, write_json)
from local_ingest.provenance import verify_evidence
from local_ingest.quality import audit_documents

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

manifest = read_json(ROOT / 'data/input_manifest.json')
print('작업 폴더:', ROOT.name)
print('입력 목록:', len(manifest), '개 파일')
print('전체 원본 재파싱:', REPARSE_FROM_ORIGINAL)
print('외부 API 호출: 0 / GitHub 작업: 0')


작업 폴더: rfp_phase2_local_20261111
입력 목록: 100 개 파일
전체 원본 재파싱: False
외부 API 호출: 0 / GitHub 작업: 0


## 2. 실제 형식과 원본 버전 확인

원본 파일을 읽어 해시를 다시 계산합니다. 확장자와 실제 형식이 다른 문서도 실제 형식으로 판독합니다.
파일이 빠졌으면 해당 원본을 복구하세요. 원본 버전이 바뀌었으면 강제 재파싱 후 평가 버전도 검토해야 합니다.
이전 정답의 해시를 새 원본의 해시로 바꾸어 맞추면 안 됩니다.


In [2]:
source_checks = []
for entry in manifest:
    row = {'doc_id': entry['doc_id'], 'file_name': entry['name']}
    try:
        path = safe_source_path(ROOT, entry['relative_path'])
        current_hash = sha256_file(path)
        kind = detect_format(path)
        row.update(exists=True, sha256=current_hash,
                   expected_sha256=entry.get('sha256'),
                   version_matches=not entry.get('sha256') or current_hash == entry['sha256'],
                   detected_format=kind,
                   extension_matches=path.suffix.lower().lstrip('.') == kind)
    except (OSError, ValueError) as error:
        row.update(exists=False, error=str(error))
    source_checks.append(row)

missing_sources = [r for r in source_checks if not r['exists']]
changed_sources = [r for r in source_checks if r.get('exists') and not r['version_matches']]
extension_mismatches = [r for r in source_checks if r.get('exists') and not r['extension_matches']]
print('실제 형식:', dict(Counter(r.get('detected_format', '판독 실패') for r in source_checks)))
print('원본 버전 일치:', sum(r.get('version_matches', False) for r in source_checks), '/', len(manifest))
print('확장자 불일치:', len(extension_mismatches))
for row in missing_sources + changed_sources + extension_mismatches:
    print(row['file_name'], row.get('error') or row.get('detected_format'))
write_json(ROOT / 'reports/source_preflight.json', source_checks)
if missing_sources:
    raise FileNotFoundError('누락되거나 판독할 수 없는 원본을 data/raw에 복구한 뒤 재실행하세요.')
if changed_sources and not REPARSE_FROM_ORIGINAL:
    raise RuntimeError('원본 버전이 달라졌습니다. 입력 목록·평가 버전을 검토하고 REPARSE_FROM_ORIGINAL=True로 재처리하세요.')


실제 형식: {'hwpx': 23, 'hwp': 47, 'pdf': 30}
원본 버전 일치: 100 / 100
확장자 불일치: 1
서울특별시_2026_2027년 재난정보관리시스템 유지관리 용역_제안요청서.hwpx hwp


## 3. 새 파싱 결과 불러오기 또는 원본 전체 재파싱

기본값은 이번 Phase 2에서 원본을 새 파서로 처리한 결과를 불러옵니다. Phase 1의 추출 텍스트를 재활용하는 경로가 아닙니다.
`REPARSE_FROM_ORIGINAL=True`이면 `force=True`로 원본 전체를 다시 읽습니다. PDF 텍스트·표를 파싱할 로컬 의존성이 필요하며 자동 설치하지 않습니다.

동봉 결과가 없거나 일부만 있거나, 원본/파서 버전과 맞지 않으면 재파싱 안내를 표시합니다.
새 문서를 추가할 때는 기존 평가 버전을 유지하면서 새 입력 목록을 별도로 만들어야 합니다.


In [3]:
def local_progress(count, total, document):
    if count == 1 or count % 10 == 0 or count == total:
        print(f"{count}/{total}: {document['file_name']} — {document['status']}")

if REPARSE_FROM_ORIGINAL:
    documents = run_ingestion(ROOT, manifest=manifest, force=True,
                              extract_pdf_tables=True, progress=local_progress)
else:
    run_manifest_path = ROOT / 'reports/run_manifest.json'
    if not run_manifest_path.is_file():
        raise FileNotFoundError('동봉 실행 기록 reports/run_manifest.json이 없습니다. 패키지를 복구하거나 REPARSE_FROM_ORIGINAL=True로 원본부터 재처리하세요.')
    recorded_run = read_json(run_manifest_path)
    recorded_manifest_hash = recorded_run.get('source_manifest_sha256')
    if recorded_manifest_hash and recorded_manifest_hash != sha256_file(ROOT / 'data/input_manifest.json'):
        raise RuntimeError('실행 기록과 현재 입력 목록의 버전이 다릅니다. REPARSE_FROM_ORIGINAL=True로 현재 목록을 재처리하세요.')
    processed_path = ROOT / 'data/processed/documents.jsonl'
    if not processed_path.is_file():
        raise FileNotFoundError('동봉 처리 결과가 없습니다. REPARSE_FROM_ORIGINAL=True로 원본부터 처리하세요.')
    saved_processed_hash = recorded_run.get('processed_files_sha256', {}).get('data/processed/documents.jsonl')
    if saved_processed_hash and saved_processed_hash != sha256_file(processed_path):
        raise RuntimeError('동봉 처리 결과가 실행 기록 이후 바뀌었습니다. REPARSE_FROM_ORIGINAL=True로 원본에서 다시 생성하세요.')
    documents = load_documents(ROOT)

if {d['doc_id'] for d in documents} != {e['doc_id'] for e in manifest}:
    raise RuntimeError('처리 결과에 누락/추가 문서가 있습니다. REPARSE_FROM_ORIGINAL=True로 전체 목록을 재처리하세요.')
source_by_id = {r['doc_id']: r for r in source_checks}
stale_results = []
for document in documents:
    if document['sha256'] != source_by_id[document['doc_id']]['sha256']:
        stale_results.append(document['doc_id'])
        continue
    kind = document.get('detected_format')
    module_path = ROOT / 'local_ingest' / (str(kind) + '.py')
    if kind and module_path.is_file() and document.get('parser_module_sha256') != sha256_file(module_path):
        stale_results.append(document['doc_id'])
if stale_results:
    raise RuntimeError(f'원본 또는 파서 버전과 다른 결과 {len(stale_results)}개. REPARSE_FROM_ORIGINAL=True로 재처리하세요.')
doc_by_id = {d['doc_id']: d for d in documents}
print('처리 상태:', dict(Counter(d['status'] for d in documents)))
print('고유 원문:', len({d['sha256'] for d in documents}), '개')
print('중복 별칭:', sum(d.get('is_duplicate_alias', False) for d in documents), '개')


처리 상태: {'parsed': 5, 'parsed_with_warnings': 95}
고유 원문: 99 개
중복 별칭: 1 개


## 4. 원문 출처 구조와 표 확인

PDF는 물리 페이지 번호, HWP는 내부 섹션·문단 번호, HWPX는 XML 섹션·문단·표 셀 좌표를 사용합니다.
HWP/HWPX 내부 위치를 인쇄 페이지 번호로 표현하지 않습니다.

XML/바이너리 레코드에서 확인한 명시 표 좌표와 PDF 선·기하정보로 추정한 표 후보를 구분합니다.
명시 좌표가 있다는 사실도 모든 표의 시각적·업무적 의미를 검수했다는 뜻은 아닙니다.


In [4]:
canonical_docs = [d for d in documents if not d.get('is_duplicate_alias')]
all_tables = [(d, t) for d in canonical_docs for t in d.get('tables', [])]
print('표 구조 분류:', dict(Counter(d.get('detected_format', '') + ':' + t.get('structure_status', 'unknown')
                                    for d, t in all_tables)))

explicit_merged = next(((d, t) for d, t in all_tables if t.get('structure_status') == 'explicit'
                       and any((c.get('row_span') or 1) > 1 or (c.get('col_span') or 1) > 1
                               for c in t.get('cells', []))), None)
if explicit_merged:
    document, table = explicit_merged
    print('명시 병합 표 예시:', document['file_name'], table['locator'])
    for cell in table['cells'][:6]:
        print({k: cell.get(k) for k in ['row', 'col', 'row_span', 'col_span', 'text', 'locator']})
pdf_candidate = next(((d, t) for d, t in all_tables if d.get('detected_format') == 'pdf'), None)
if pdf_candidate:
    document, table = pdf_candidate
    print('PDF 표 후보:', document['file_name'], table['locator'], table.get('structure_status'))
    print('좌표:', table.get('bbox'), '방법:', table.get('method'))
    print('앞쪽 행:', table['rows'][:2])


표 구조 분류: {'hwpx:explicit': 2845, 'hwp:explicit': 3876, 'pdf:heuristic': 3352, 'pdf:unresolved': 248, 'hwp:unresolved': 1}
명시 병합 표 예시: 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx Contents/section0.xml / table 2
{'row': 0, 'col': 0, 'row_span': 2, 'col_span': 1, 'text': '담 당', 'locator': 'Contents/section0.xml / table 2 / row 0 col 0'}
{'row': 0, 'col': 1, 'row_span': 1, 'col_span': 1, 'text': '성  명', 'locator': 'Contents/section0.xml / table 2 / row 0 col 1'}
{'row': 0, 'col': 2, 'row_span': 1, 'col_span': 1, 'text': '소  속', 'locator': 'Contents/section0.xml / table 2 / row 0 col 2'}
{'row': 0, 'col': 3, 'row_span': 1, 'col_span': 1, 'text': '전화번호', 'locator': 'Contents/section0.xml / table 2 / row 0 col 3'}
{'row': 0, 'col': 4, 'row_span': 1, 'col_span': 1, 'text': 'E-mail', 'locator': 'Contents/section0.xml / table 2 / row 0 col 4'}
{'row': 1, 'col': 1, 'row_span': 1, 'col_span': 1, 'text': '장희수', 'locator': 'Contents/section0.xml / table 2 / row 1 col 1'}
PDF 표 후보: 한국기계연구원_2026_통합정보시

## 5. Phase 1 정답 근거를 새 원문에서 다시 찾기

정답의 문서 ID와 원본 해시를 확인한 뒤 인용 문자열을 찾습니다.
완전 일치와 공백만 정규화한 일치는 따로 집계합니다. 숫자·기호를 고쳐 맞추지 않습니다.
같은 문구가 여러 위치에 있으면 후보를 보존하고 임의의 한 위치로 확정하지 않습니다.

HWPX의 `reference_blocks`는 원본 XML에서 재구성한 **인용 검증 전용 섹션 뷰**입니다.
일반 `blocks`와 함께 다시 인덱싱하면 내용이 중복됩니다. 다음 청킹 입력은 `indexable_blocks.jsonl`을 사용합니다.


In [5]:
dev_cases = read_jsonl(ROOT / 'eval/questions.dev.jsonl')
holdout_cases = read_jsonl(ROOT / 'eval/questions.holdout.jsonl')
cases = dev_cases + holdout_cases
evidence_report = verify_evidence(documents, cases)
write_json(ROOT / 'reports/evidence_validation.json', evidence_report)
summary_keys = ['total', 'exact_matches', 'normalized_matches', 'unmatched',
                'hash_mismatch', 'resolved_matches', 'ambiguous_matches',
                'at_original_locator', 'relocated_matches', 'reference_view_matches']
print(json.dumps({k: evidence_report[k] for k in summary_keys}, ensure_ascii=False, indent=2))
print(evidence_report['verification_scope'])


{
  "total": 684,
  "exact_matches": 680,
  "normalized_matches": 4,
  "unmatched": 0,
  "hash_mismatch": 0,
  "resolved_matches": 683,
  "ambiguous_matches": 1,
  "at_original_locator": 684,
  "relocated_matches": 0,
  "reference_view_matches": 100
}
새 파싱 원문의 해시·인용 복구·원문 위치 검증. 의미 지원 여부와 실제 RAG 답변 성능 평가는 별도.


## 6. 검증한 인용을 원문 위치로 역추적

아래 예시는 개발용 사례에서 단일 위치로 확인된 근거 한 건입니다.
문자 시작·끝 위치는 정규화 전 원문 블록의 Python 문자 인덱스입니다(`char_end` 미포함).
문서 이름·해시·위치·실제 인용 구절을 함께 보관합니다.


In [6]:
dev_ids = {c['case_id'] for c in dev_cases}
example = next((r for r in evidence_report['records']
                if r['case_id'] in dev_ids and r['location_resolved'] and not r['reference_view']), None)
if example is None:
    example = next((r for r in evidence_report['records']
                    if r['case_id'] in dev_ids and r['location_resolved']), None)
if example:
    document = doc_by_id[example['source_doc_id']]
    views = document.get('blocks', []) + document.get('reference_blocks', [])
    source_block = next(b for b in views if b['block_id'] == example['block_id'])
    recovered = source_block['text'][example['char_start']:example['char_end']]
    assert recovered == example['matched_text']
    print('사례:', example['case_id'])
    print('문서:', example['file_name'])
    print('원본 SHA256:', example['source_sha256'])
    print('위치:', example['current_locator'], example['char_start'], example['char_end'])
    print('복구 방법:', example['match_mode'])
    print('근거:', recovered[:500])
else:
    print('단일 위치로 확인된 개발용 근거가 없습니다. 검증 보고서의 후보·미복구 항목을 확인하세요.')


사례: C001
문서: 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp
원본 SHA256: 8c293ac2bb077438c0aaa974746d203a9196ec3eaa0d2c7971e5f6d3e4a28e28
위치: BodyText/Section0 / paragraph 99 0 23
복구 방법: exact
근거: □ 사업예산 : 688백만원 (부가세포함)


## 7. 시각 검수 전사와 자동 추출 구분

일부 PDF의 숫자는 글자가 아닌 벡터 윤곽선으로 저장되어 일반 텍스트 추출에서 빠질 수 있습니다.
패키지의 검수 전사는 실제 원본 페이지를 보고 기록한 별도 보완 출처입니다.
원본 해시·페이지·좌표·`visual_verified` 상태가 확인된 전사만 적용하며 자동 추출 성공 건수에 더하지 않습니다.
OCR 결과는 이 검수 전사로 자동 승격하지 않습니다.


In [7]:
transcriptions = read_json(ROOT / 'data/reviewed_transcriptions.json')
applied_transcriptions = [(d, b) for d in documents for b in d.get('supplemental_blocks', [])]
print('시각 검수 전사 기록:', len(transcriptions), '개 / 적용된 별도 블록:', len(applied_transcriptions), '개')
for entry in transcriptions:
    document = doc_by_id[entry['doc_id']]
    assert entry['review_status'] == 'visual_verified'
    assert entry['sha256'] == document['sha256']
    assert len(entry['bbox']) == 4
    matching = [b for b in document.get('supplemental_blocks', [])
                if b['block_id'] == 'transcription_' + entry['check_id']]
    assert len(matching) == 1 and matching[0]['text'] == entry['text']
    assert matching[0]['source_method'] == 'reviewed_visual_transcription'
    print(entry['check_id'], 'PDF 물리 페이지', entry['page'],
          '— 자동 추출과 분리한 시각 검수 전사')


시각 검수 전사 기록: 3 개 / 적용된 별도 블록: 3 개
KSD_P3_NAME PDF 물리 페이지 3 — 자동 추출과 분리한 시각 검수 전사
KSD_P3_DURATION PDF 물리 페이지 3 — 자동 추출과 분리한 시각 검수 전사
KSD_P3_BUDGET PDF 물리 페이지 3 — 자동 추출과 분리한 시각 검수 전사


## 8. 품질 경고와 남은 검수 목록

성공·실패, 버전 불일치, 명시 표 좌표의 무결성, 숫자 누락 의심 등을 점검합니다.
경고 목록은 문서 처리 실패나 모델 오답 점수와 동일하지 않습니다.
시각 검수한 특정 표 행·전사 구간의 회귀검사도 전체 표의 정확도로 확대하지 않습니다.


In [8]:
quality_report = audit_documents(documents, evidence_report=evidence_report, root=ROOT)
quality_keys = ['document_files', 'unique_contents', 'successful_files', 'failed_files',
                'canonical_text_blocks', 'pdf_physical_pages_unique_contents',
                'reviewed_transcription_blocks', 'review_queue_entries', 'review_queue_open_entries',
                'all_input_sha256_match', 'all_source_locations_present',
                'phase2_implementation_ready', 'visual_or_ocr_review_items_remain']
print(json.dumps({k: quality_report.get(k) for k in quality_keys}, ensure_ascii=False, indent=2))
print('경고 유형:', quality_report['warning_counts'])
print('숫자 공란 경고 원문 검수:', quality_report.get('numeric_priority_review', {}))
review_queue = quality_report.get('review_queue', [])
print('검수 목록 일부:')
for row in review_queue[:10]:
    print(row['priority'], row['file_name'], row.get('locator'), row['code'])
for name in ['visual_transcription_regression', 'table_sample_regression']:
    check = quality_report.get(name, {})
    print(name, {k:v for k,v in check.items() if k != 'records'})


{
  "document_files": 100,
  "unique_contents": 99,
  "successful_files": 100,
  "failed_files": 0,
  "canonical_text_blocks": 182239,
  "pdf_physical_pages_unique_contents": 2872,
  "reviewed_transcription_blocks": 3,
  "review_queue_entries": 1321,
  "review_queue_open_entries": 1304,
  "all_input_sha256_match": true,
  "all_source_locations_present": true,
  "phase2_implementation_ready": true,
  "visual_or_ocr_review_items_remain": true
}
경고 유형: {'NON_TEXT_OBJECTS_PRESENT': 76, 'HWP_LAYOUT_NOT_RENDERED': 47, 'EXTENSION_FORMAT_MISMATCH': 1, 'PDF_VECTOR_GLYPH_REVIEW': 997, 'PDF_NUMERIC_VALUE_UNRESOLVED': 17, 'PDF_LOW_TEXT_PAGE': 25, 'PDF_OCR_REVIEW_NEEDED': 16, 'PDF_TABLE_GRID_UNRESOLVED': 12, 'PDF_TABLES_HEURISTIC': 30, 'INVALID_UTF16_REPLACED': 1, 'TABLE_COORDINATES_UNRESOLVED': 1, 'PDF_TEXT_CONTROL_OR_REPLACEMENT': 4, 'VISUAL_TRANSCRIPTION_APPLIED': 1}
숫자 공란 경고 원문 검수: {'flagged_pages': 17, 'version_matched_reviewed_pages': 17, 'unreviewed_pages': 0, 'classification_counts': {'blan

## 9. 선택: PDF 원본 페이지를 로컬 이미지로 보기

기본값은 실행하지 않습니다. `RUN_PDF_RENDER=True`로 바꾸면 선택한 원본의 물리 페이지를 이미지로 저장합니다.
아래 기본 대상은 숫자 벡터 윤곽선 문제가 확인된 문서의 3페이지입니다.
이 기능은 로컬 `PyMuPDF`가 필요하며 설치나 외부 전송을 자동으로 수행하지 않습니다.


In [9]:
REVIEW_DOC_ID = '1XuHMGElCN1wjEBBucRF55JWKMCePUmUM'
REVIEW_PHYSICAL_PAGE = 3
if RUN_PDF_RENDER:
    try:
        import fitz
    except ImportError as error:
        raise RuntimeError('선택적 PDF 렌더에는 로컬 PyMuPDF가 필요합니다. 자동 설치하지 않습니다.') from error
    target = doc_by_id[REVIEW_DOC_ID]
    source_path = safe_source_path(ROOT, target['relative_path'])
    with fitz.open(source_path) as pdf:
        if not 1 <= REVIEW_PHYSICAL_PAGE <= len(pdf):
            raise ValueError('선택한 물리 페이지가 문서 범위를 벗어납니다.')
        pixmap = pdf[REVIEW_PHYSICAL_PAGE - 1].get_pixmap(matrix=fitz.Matrix(2,2), alpha=False)
        image_path = ROOT / 'reports/local_views' / f'{REVIEW_DOC_ID}_p{REVIEW_PHYSICAL_PAGE}.png'
        image_path.parent.mkdir(parents=True, exist_ok=True)
        pixmap.save(image_path)
    print('원본 페이지 이미지:', image_path)
    try:
        from IPython.display import Image, display
    except ImportError:
        pass
    else:
        display(Image(filename=str(image_path)))
else:
    print('선택적 PDF 렌더를 실행하지 않았습니다.')


선택적 PDF 렌더를 실행하지 않았습니다.


## 10. 선택: 로컬 숫자 OCR 후보 점검

기본값은 실행하지 않습니다. `RUN_LOCAL_OCR=True`일 때만 선택 페이지의 작은 벡터 자형 후보를 조사하고 로컬 `tesseract`로 숫자 인식을 시도합니다.
`PyMuPDF`, 로컬 `tesseract`와 영문 숫자 인식 데이터가 필요하며 자동 설치하지 않습니다.
이 방법은 전체 페이지 OCR이나 한글 문맥 복원이 아닙니다.

결과는 항상 `unreviewed` 보완 자료로 따로 저장합니다. 추출 본문·검수 전사·인덱싱 파일을 덮어쓰지 않으며 숫자값을 확정하지 않습니다.


In [10]:
if RUN_LOCAL_OCR:
    from local_ingest.ocr import inspect_vector_regions
    target = doc_by_id[REVIEW_DOC_ID]
    source_path = safe_source_path(ROOT, target['relative_path'])
    ocr_result = inspect_vector_regions(source_path, pages=[REVIEW_PHYSICAL_PAGE],
                                        max_regions=30, run_ocr=True)
    ocr_result.update(doc_id=target['doc_id'], sha256=sha256_file(source_path),
                      review_status='unreviewed')
    assert all(b['review_status'] == 'unreviewed' for b in ocr_result['blocks'])
    ocr_path = ROOT / 'reports/local_ocr' / f'{REVIEW_DOC_ID}_p{REVIEW_PHYSICAL_PAGE}_unreviewed.json'
    write_json(ocr_path, ocr_result)
    print('미검수 OCR 후보:', len(ocr_result['blocks']), '개')
    print('경고:', ocr_result['warnings'])
    print('별도 보완 자료:', ocr_path)
else:
    print('선택적 로컬 OCR을 실행하지 않았습니다.')


선택적 로컬 OCR을 실행하지 않았습니다.


## 11. Phase 2 완료 기준과 청킹 준비

문서별 처리 상태·버전·출처가 기록되고 평가 근거를 복구할 수 있어야 합니다.
남은 시각/OCR 검수는 구분해 유지합니다. 문서 전체·그림·모든 표의 의미를 완벽히 보존했다고 표시하지 않습니다.

다음 청킹 입력은 `data/processed/indexable_blocks.jsonl`입니다.
동일 내용의 사본과 인용 검증용 `reference_blocks`는 중복 인덱싱하지 않습니다.
시각 검수 전사는 출처 방법을 유지하고, 미검수 OCR 후보는 일반 근거로 자동 편입하지 않습니다.


In [11]:
transcription_check = quality_report.get('visual_transcription_regression', {})
table_check = quality_report.get('table_sample_regression', {})
completion_checks = {
    '입력 파일별 처리 결과 기록': len(documents) == len(manifest),
    '전체 원본 판독 성공': quality_report['failed_files'] == 0,
    '원본 버전 일치': quality_report['all_input_sha256_match'],
    '원문 위치 정보 보존': quality_report['all_source_locations_present'],
    '명시 표 좌표 무결성': not quality_report['source_structure_issues'],
    '평가 인용 복구': evidence_report['unmatched'] == 0,
    '별도 전사 검수 구간 회귀검사': transcription_check.get('all_covered', not transcriptions),
    '선정한 표 행 회귀검사': table_check.get('all_passed', True),
}
print(json.dumps(completion_checks, ensure_ascii=False, indent=2))
print('구현·회귀검사 기준 충족:', all(completion_checks.values()))
print('검수 기록 전체:', len(review_queue), '건 / 추가 검수 항목:', quality_report.get('review_queue_open_entries', len(review_queue)), '건')
print('단일 위치로 임의 확정하지 않은 인용:', evidence_report['ambiguous_matches'], '건')
assert all(completion_checks.values()), '미충족 항목과 reports/의 세부 결과를 확인한 뒤 재처리·검수하세요.'
if REPARSE_FROM_ORIGINAL:
    from run_phase2 import record_run
    record_run(ROOT, documents, quality_report, evidence_report)
    print('현재 재파싱 결과의 버전·출력 해시를 새 실행 기록에 저장했습니다.')
print('다음 입력:', ROOT / 'data/processed/indexable_blocks.jsonl')


{
  "입력 파일별 처리 결과 기록": true,
  "전체 원본 판독 성공": true,
  "원본 버전 일치": true,
  "원문 위치 정보 보존": true,
  "명시 표 좌표 무결성": true,
  "평가 인용 복구": true,
  "별도 전사 검수 구간 회귀검사": true,
  "선정한 표 행 회귀검사": true
}
구현·회귀검사 기준 충족: True
검수 기록 전체: 1321 건 / 추가 검수 항목: 1304 건
단일 위치로 임의 확정하지 않은 인용: 1 건
다음 입력: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase2_local_20261111/data/processed/indexable_blocks.jsonl
